<img src="https://iteso.mx/documents/27014/202031/Logo-ITESO-MinimoH.png"
     align="right"
     width="300"/>

# Práctica Modelos GARCH

## *Modelos no lineales para pronósitico*  - Pedro Martinez

---

<img src="https://corporate.exxonmobil.com/-/media/global/icons/logos/exxonmobillogocolor2x.png"
     align="left"
     width="300"/>

<p align="justify">
 Para este ejercicio, utilizaremos la empresa ExxonMobil Corporation (Ticker: XOM). Es una de las mayores empresas petroleras y de gas del mundo, involucrada en la exploración, producción, transporte y venta de petróleo crudo y gas natural.
</p>
<p align="justify">
Ante una crisis energética o volatilidad en los precios del barril, sus acciones experimentan lo que en series de tiempo llamamos agrupamiento de volatilidad (volatility clustering).
</p>



In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf, pacf
from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## **<font color= #0077b6> Obtenemos datos y gráficas de serie de tiempo </font>**
Obtenemos la serie de tiempo de la empresa utilizando yfinance para graficar. Se calcula los rendimientos logarítmicos porcentuales, los cuales se representan como:$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) \times 100$$

Usualmente se usan retorno logaritmico porque en escalas pequeñas su compertamiento es muy similar al del retorno "normal" y al ser logaritmo, el comportamiento sigue una distribución más estable.

In [4]:
# Descargar datos de ExxonMobil
ticker = 'XOM'
xom = yf.Ticker(ticker)
df = xom.history(start='2020-01-01', end='2026-03-08')

# Calculamos los retornos logarítmicos porcentuales usando 'Close'
returns = 100 * np.log(df['Close'] / df['Close'].shift(1)).dropna()

# Gráfica de la serie de retornos usando Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines', name='Retornos XOM'))
fig.update_layout(title=f'Retornos Diarios de {ticker}',
                  yaxis_title='Retornos (%)')
fig.show()

## **<font color= #0077b6> Selección y validación de modelo. </font>**

Para justificar si es estadísticamente viable utilizar un modelo GARCH, se calculan las graficas ACF y PACF; pero utilizando los retornos al cuadrado.

**Retornos vs. Retornos al Cuadrado**

Las gráficas ACF y PACF miden la autocorrelación (qué tanto se parece el dato de hoy vs los datos del pasado)

- ACF/PACF de los Retornos Normales ($r_t$)
  - ¿Qué miden? ->
    Dependencia lineal en el primer momento estadístico (la media).
  - ¿Qué pregunta responden? ->
    Saber si la acción subió ayer me ayuda a predecir si subirá hoy.
  - Lo que verás en la gráfica ->
    La gráfica de los retornos normales $r_t$ casi no mostrará barras rojas significativas. Esto indica que el retorno diario es prácticamente ruido blanco (impredecible).
    
- ACF/PACF de los Retornos al Cuadrado ($r_t^2$)
  - ¿Qué miden? ->
    Dependencia no lineal en el segundo momento estadístico (la varianza/volatilidad). Al elevar al cuadrado, eliminamos el signo (positivo o negativo) y nos quedamos solo con la magnitud del movimiento.
  - ¿Qué pregunta responden? ->
    Saber que ayer hubo un movimiento violento (sin importar si fue hacia arriba o hacia abajo) me ayuda a predecir si hoy el mercado seguirá turbulento.
  - Lo que verás en la gráfica ->
    Verás múltiples barras que sobresalen de la zona de significancia. Esto demuestra matemáticamente el Volatility Clustering (agrupamiento de volatilidad).

Si la PACF o ACF de retornos al cuadrado no muestra barras significativas, no se recomienda utilizar un modelo GARCH (Puede ser un modelo EGARCH o de otra familia)

In [5]:
# Retornos al cuadrado
sq_returns = returns**2

# Calcular ACF y PACF
lag_acf = acf(sq_returns, nlags=20)
lag_pacf = pacf(sq_returns, nlags=20, method='ols')

# Graficar ACF y PACF con Plotly
fig = make_subplots(rows=1, cols=2, subplot_titles=('ACF de Retornos al Cuadrado', 'PACF de Retornos al Cuadrado'))

# Añadir barras de ACF
fig.add_trace(go.Bar(x=np.arange(len(lag_acf)), y=lag_acf, name='ACF'), row=1, col=1)
# Añadir barras de PACF
fig.add_trace(go.Bar(x=np.arange(len(lag_pacf)), y=lag_pacf, name='PACF'), row=1, col=2)

# Líneas de significancia (aprox 1.96 / sqrt(N))
sig_level = 1.96 / np.sqrt(len(returns))
for i in [1, 2]:
    fig.add_hline(y=sig_level, line_dash="dash", line_color="red", row=1, col=i)
    fig.add_hline(y=-sig_level, line_dash="dash", line_color="red", row=1, col=i)

fig.update_layout(title='Análisis de Dependencia de Varianza', showlegend=False)
fig.show()

## **<font color= #0077b6> Selección de hiperparametros del modelo. </font>**



Hay muchas maneras de "estimar" el orden del modelo GARCH, sin embargo, la recomendación es seguir el principio de Parsimonia o de la navaja de Ockham

- La librería más utilizada en python es `ARCH`, porque puede modelar ARCH y GARCH, eso se especifica en el parametro `vol`. (Aparte de que es Gratis)

- Cuando se declara un modelo hay que indicar que distribución tienen los retornos. Usualmente, la distribución tiene Leptocurtosis o "colas pesadas".
    - `dist='normal'`: Distribución normal estándar (peligrosa para crisis).

    - `dist='t'`: t-Student La distribución normal se "aplasta" en el centro y hace que las colas sean más gruesas, asignando mayor probabilidad matemática a las caídas extremas

    - `dist='skewt'`: t-Student sesgada. No solo tiene colas pesadas, sino que asume que la cola izquierda (pérdidas) es más pesada que la derecha (ganancias).

    - `dist='ged'`: Distribución de Error Generalizado.

- Cuantil empírico para VaR (Value at Risk). $VaR_{t} = \hat{\mu} + \hat{\sigma}_t \cdot Z_{\alpha}$
  - La Media Condicional ($\mu$): Es el rendimiento esperado en un día normal.
  - La Volatilidad Condicional ($\hat{\sigma}_t$): Es lo que predecimos con el modelo GARCH ¿El mercado está tranquilo (volatilidad de 1%) o en pánico por la crisis petrolera (volatilidad del 4%)
  - El Cuantil ($Z_{\alpha}$): Este es el "factor de castigo" probabilístico. En una distribución Normal o t-Student, el cuantil del 5% es un número negativo (ej. -1.64 o -2.10). Representa a cuántas desviaciones estándar de distancia de la media te tienes que alejar para estar en el peor 5% de los casos.


In [6]:
# Ajuste del modelo GARCH(1,1)
am = arch_model(returns, vol='Garch', p=1, q=1, dist='t')
res = am.fit(disp='off')


# Extraemos la volatilidad condicional modelada (Varianza que predijimos)
cond_vol = res.conditional_volatility

# Cálculo del Value at Risk (VaR) Histórico Condicional (GARCH) al 5%
# Cuantil empírico al 5% de los residuales estandarizados
q_5 = res.std_resid.quantile(0.05)

# VaR = Media Condicional + (Cuantil * Volatilidad Condicional)
# GARCH por defecto asume media constante, la extraemos de los parámetros
mean = res.params['mu']
VaR_5 = mean + cond_vol * q_5

# 5. Graficar los Retornos vs el VaR predictivo
fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns.squeeze(), mode='lines',
                         name='Retornos Reales', opacity=0.6))
fig.add_trace(go.Scatter(x=VaR_5.index, y=VaR_5, mode='lines',
                         name='VaR 5% (GARCH)', line=dict(color='red')))

fig.update_layout(title='Backtesting de Riesgo: Retornos de XOM vs GARCH(1,1) VaR al 5%',
                  yaxis_title='Retornos (%)')
fig.show()

## **<font color= #0077b6> Interpretación del summary. </font>**
- Mean model indica cual es el promedio del retorno esperado.
- Volatility model indica los parámetros que definen la ecuación de la varianza: $\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$.

  - Omega($\omega$):Es la varianza base a largo plazo. Es el nivel de "ruido de fondo" que tiene la acción cuando no hay crisis.
  - alpha[1] ($\alpha$): Es el Efecto ARCH. Mide qué tan sensible es la volatilidad de mañana a los "choques" (noticias o eventos inesperados) de hoy. Valores cercanos a 0 indican poca reactividad.
  - beta[1] ($\beta$):  Es el Efecto GARCH. Mide la "memoria" o persistencia de la volatilidad. Un valor cercano a 1 significa que si hoy hay pánico en el mercado, mañana casi seguro también lo habrá y tardará en calmarse.
- Distribución indica como se comportan los rendimientos.
  - nu ($\nu$): Son los grados de libertad de la distribución t-Student. Una distribución Normal equivale matemáticamente a una t-Student con grados de libertad infinitos ($\nu = \infty$). Cuando los grados son bajos, quiere decir que existe la posibilidad de "colas pesadas".

In [7]:
print(res.summary())

                        Constant Mean - GARCH Model Results                         
Dep. Variable:                        Close   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -3111.04
Distribution:      Standardized Student's t   AIC:                           6232.07
Method:                  Maximum Likelihood   BIC:                           6258.81
                                              No. Observations:                 1551
Date:                      Wed, Mar 11 2026   Df Residuals:                     1550
Time:                              20:22:30   Df Model:                            1
                                 Mean Model                                
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
mu     